In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display
import re, json

In [ ]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
MODEL = "llama3.2:1b"

good_guy_system_pmt = """
You are a person with good character.
you have below personality traits,
Calm, intelligent, and practical
Shows occasional compassion and a code of honor
Speaks little; observes a lot
Manipulative when needed, but avoids unnecessary cruelty
More of an anti-hero than a pure hero
"""

bad_guy_system_pmt = """
You are a person with bad character.
you have below personality traits,
Cold, ruthless, and highly professional
Efficient and disciplined
Has almost no empathy
Carries out jobs with brutal consistency
Driven by money and self-interest
"""

ugly_guy_system_pmt = """
You are a person with ugly character.
you have below personality traits,
Chaotic, emotional, loud, and unpredictable
Greedy and selfish, but surprisingly human
Funny and clever
Frequently switches between desperation, anger, and loyalty
Morally messy rather than evil
"""

In [ ]:
good_guy_messages = ["Let’s split the gold fairly."]
bad_guy_messages = ["Fairly? That’s a word people use before getting robbed."]
ugly_guy_messages = ["You two keep talking morality — I’m busy counting shares that somehow belong to me."]

In [ ]:
def clean_dialogue(text):
    text = re.sub(r"\([^)]*\)", "", text)
    text = re.sub(r"\*[^*]*\*", "", text)
    text = re.sub(r'^["\']|["\']$', "", text)
    text = re.sub(r'["\']([^"\']*)["\']', r"\1", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [ ]:
def build_conversation_history():
#     [
#   {
#     "speaker": "Good",
#     "message": "Let's split the gold fairly"
#   },
#   {
#     "speaker": "Bad",
#     "message": "Fairly? That's for fools"
#   },
#   {
#     "speaker": "Ugly",
#     "message": "I'm taking all of it"
#   },
#   {
#     "speaker": "Ugly",
#     "message": "And the horses too"
#   }
# ]
    conversation = []
    max_len = max(len(good_guy_messages), len(bad_guy_messages), len(ugly_guy_messages))

    for i in range(max_len):
        if i < len(good_guy_messages):
            conversation.append({"speaker": "Good", "message": good_guy_messages[i]})
        if i < len(bad_guy_messages):
            conversation.append({"speaker": "Bad", "message": bad_guy_messages[i]})
        if i < len(ugly_guy_messages):
            conversation.append({"speaker": "Ugly", "message": ugly_guy_messages[i]})

    return json.dumps(conversation, indent=2)

In [ ]:
def call_good_guy():
    conversation = build_conversation_history()
    user_prompt = f"""
    You are Good, in conversation with Bad and Ugly.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Good."""

    messages = [
        {"role": "system", "content": good_guy_system_pmt},
        {"role": "user", "content": user_prompt},
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
        max_tokens=50
    )

    content = ""
    for chunk in response:
        if chunk.choices[0].delta.content:
            content += chunk.choices[0].delta.content

    # Clean up any stage directions that slipped through
    return clean_dialogue(content)

In [ ]:
def call_bad_guy():
    conversation = build_conversation_history()
    user_prompt = f"""
    You are Bad, in conversation with Good and Ugly.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Bad."""

    messages = [
        {"role": "system", "content": bad_guy_system_pmt},
        {"role": "user", "content": user_prompt},
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
        max_tokens=50
    )

    content = ""
    for chunk in response:
        if chunk.choices[0].delta.content:
            content += chunk.choices[0].delta.content

    # Clean up any stage directions that slipped through
    return clean_dialogue(content)

In [ ]:
def call_ugly_guy():
    conversation = build_conversation_history()
    user_prompt = f"""
    You are Ugly, in conversation with Good and Bad.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Ugly."""

    messages = [
        {"role": "system", "content": ugly_guy_system_pmt},
        {"role": "user", "content": user_prompt},
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True,
        max_tokens=50
    )

    content = ""
    for chunk in response:
        if chunk.choices[0].delta.content:
            content += chunk.choices[0].delta.content

    # Clean up any stage directions that slipped through
    return clean_dialogue(content)

In [ ]:
#call_gemma()

In [ ]:
display(Markdown(f"### Good:\n{good_guy_messages[-1]}\n"))
display(Markdown(f"### Bad:\n{bad_guy_messages[-1]}\n"))
display(Markdown(f"### Ugly:\n{ugly_guy_messages[-1]}\n"))

for i in range(5):
    good_next = call_good_guy()
    display(Markdown(f"### Good:\n{good_next}\n"))
    good_guy_messages.append(good_next)

    bad_next = call_bad_guy()
    display(Markdown(f"### Bad:\n{bad_next}\n"))
    bad_guy_messages.append(bad_next)

    ugly_next = call_ugly_guy()
    display(Markdown(f"### Ugly:\n{ugly_next}\n"))
    ugly_guy_messages.append(ugly_next)

